# netMHCpan
![netMHCpan](https://proto-bio.github.io/proto-assets/images/tool/netmhcpan/hero.png)

This notebook demonstrates the two netMHCpan tools for MHC class I epitope prediction.
`run_netmhcpan_peptides` scores peptides you already have against a panel of MHC class I
alleles, and `run_netmhcpan_protein` windows a protein into class I k-mers and scores every
window, which is the usual way to screen a designed or natural sequence for epitopes.
netMHCpan 4.1 is an ensemble of neural networks trained jointly on binding-affinity (BA) and
mass-spectrometry eluted-ligand (EL) data, so a single pan-specific model covers any allele
with a known sequence. Predictions come back as a %rank against a background of random
peptides, which is what makes them comparable across alleles.

Learn more at [services.healthtech.dtu.dk/services/NetMHCpan-4.1](https://services.healthtech.dtu.dk/services/NetMHCpan-4.1/)
and in the method paper: Reynisson, Alvarez, Paul, Peters & Nielsen, *NetMHCpan-4.1 and
NetMHCIIpan-4.0: improved predictions of MHC antigen presentation by concurrent motif
deconvolution and integration of MS MHC eluted ligand data*, Nucleic Acids Research 48(W1):W449-W454 (2020),
[doi:10.1093/nar/gkaa379](https://doi.org/10.1093/nar/gkaa379).

**Note:** netMHCpan is license-gated and user-provisioned. It is not bundled with proto-tools
and cannot be downloaded automatically: the DTU academic license requires you to register and
accept the terms yourself, and forbids redistributing the package. To run the cells below you
must install netMHCpan 4.1 locally and either set the `NETMHCPAN_HOME` environment variable to
the install root or pass `netmhcpan_dir` in the config. See the toolkit's `SETUP.md` for the
provisioning steps. Every run cell in this notebook errors without a local install.

In [1]:
from proto_tools.tools import (
    NetMHCpanPeptidesConfig,
    NetMHCpanPeptidesInput,
    NetMHCpanProteinConfig,
    NetMHCpanProteinInput,
    run_netmhcpan_peptides,
    run_netmhcpan_protein,
)
from proto_tools.utils.notebook_docs import display_api_reference

In [2]:
# Display input API reference
display_api_reference("netmhcpan-peptides", "input", "run_netmhcpan_peptides")

# Four well-characterised class I peptides:
#   AAAWYLWEV - HIV-1 reverse transcriptase, a known HLA-A*02:01 binder
#   GILGFVFTL - influenza A M1(58-66), the canonical HLA-A*02:01 epitope
#   KLGGALQAK - human CMV IE1(316-324), HLA-A*03:01-restricted (an A*02:01 non-binder)
#   AAGLQDCTM - no reported class I restriction; included as a negative control
inputs = NetMHCpanPeptidesInput(
    peptides=["AAAWYLWEV", "GILGFVFTL", "KLGGALQAK", "AAGLQDCTM"],
)

**Input** — `NetMHCpanPeptidesInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>peptides</code> | <code>list[str]</code> | required | MHC class I peptides (8-14 residues) to score against every allele. |

In [3]:
# Display config API reference
display_api_reference("netmhcpan-peptides", "config", "run_netmhcpan_peptides")

# A three-allele panel. netMHCpan itself accepts only the unstarred spelling, but the wrapper
# normalizes the starred forms ("HLA-A*02:01", "A*02:01") and bare human loci ("A02:01") too,
# so all the spellings in common use converge on the same allele.
config = NetMHCpanPeptidesConfig(
    alleles=["HLA-A*02:01", "HLA-A01:01", "HLA-B07:02"],
)
print(f"Normalized allele panel: {config.alleles}")

**Config** — `NetMHCpanPeptidesConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>alleles</code> | <code>list[str]</code> | <code>['HLA-A02:01']</code> | MHC class I alleles to predict against, e.g. ['HLA-A02:01', 'HLA-B07:02']. |
| <code>include_binding_affinity</code> | <code>bool</code> | <code>False</code> | Also predict affinity (-BA): adds BA score, BA %rank and nM affinity. |
| <code>strong_rank</code> | <code>float</code> | <code>0.5</code> | EL %rank at or below which a peptide counts as a strong binder. |
| <code>weak_rank</code> | <code>float</code> | <code>2.0</code> | EL %rank at or below which a peptide counts as a weak binder. |
| <code>netmhcpan_dir</code> | <code>str &#124; None</code> | <code>None</code> | Local netMHCpan install root; overrides $NETMHCPAN_HOME and the tool cache. |

Normalized allele panel: ['HLA-A02:01', 'HLA-A01:01', 'HLA-B07:02']


In [4]:
# Display output API reference
display_api_reference("netmhcpan-peptides", "output", "run_netmhcpan_peptides")

**Output** — `NetMHCpanPeptidesOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>predictions</code> | <code>list[PeptidePrediction]</code> | required | One entry per input peptide, index-aligned with the inputs. |
| <code>netmhcpan_version</code> | <code>str &#124; None</code> | <code>None</code> | Version reported by the invoked binary; results differ across versions. |

**Metrics** — `NetMHCpanMetrics` (one set per `predictions` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>strong_binder_count</code> | <code>int</code> | <code>&gt;= 0</code> |  | Peptide-allele pairs at or below the strong %rank threshold. |
| <code>weak_binder_count</code> | <code>int</code> | <code>&gt;= 0</code> |  | Pairs between the strong and weak %rank thresholds. |
| <code>strong_binder_fraction</code> | <code>float</code> | <code>[0, 1]</code> |  | Strong binders over all pairs; comparable across panels and lengths. |
| <code>best_rank</code> | <code>float</code> | <code>[0, 100]</code> | <code>percent</code> | Lowest EL %rank over all pairs; the single strongest predicted binder. |
| <code>binding_propensity</code> **(primary)** | <code>float</code> | <code>[0, 1]</code> |  | Mean smooth 0-1 propensity over pairs; a continuous optimization target. |
| <code>best_rank_ba</code> | <code>float</code> | <code>[0, 100]</code> | <code>percent</code> | Lowest binding-affinity %rank over all pairs. |
| <code>best_affinity_nm</code> | <code>float</code> | <code>&gt;= 0</code> | <code>nM</code> | Tightest predicted affinity over all pairs; lower binds more tightly. |

In [5]:
# Score every peptide against every allele in the panel (one netMHCpan invocation).
peptide_result = run_netmhcpan_peptides(inputs, config)
print(f"netMHCpan version: {peptide_result.netmhcpan_version}")

# One PeptidePrediction per input peptide, index-aligned with inputs.peptides. Each carries
# one binding record per configured allele, plus that peptide's summary metrics.
for prediction in peptide_result.predictions:
    print(f"\n{prediction.peptide}")
    for binding in prediction.bindings:
        print(
            f"  {binding.allele:<12} rank_el={binding.rank_el:>7.3f}  "
            f"score_el={binding.score_el:.6f}  {binding.binder_class}"
        )
    print(
        f"  summary: strong={prediction.strong_binder_count} weak={prediction.weak_binder_count} "
        f"best_rank={prediction.best_rank:.3f} binding_propensity={prediction.binding_propensity:.4f}"
    )

Running run_netmhcpan_peptides [00:00]

netMHCpan version: 4.1b

AAAWYLWEV
  HLA-A02:01   rank_el=  0.472  score_el=0.440383  strong
  HLA-A01:01   rank_el= 10.140  score_el=0.002824  non-binder
  HLA-B07:02   rank_el= 16.477  score_el=0.000822  non-binder
  summary: strong=1 weak=0 best_rank=0.472 binding_propensity=0.3331

GILGFVFTL
  HLA-A02:01   rank_el=  0.114  score_el=0.789948  strong
  HLA-A01:01   rank_el= 25.341  score_el=0.000590  non-binder
  HLA-B07:02   rank_el= 10.849  score_el=0.002070  non-binder
  summary: strong=1 weak=0 best_rank=0.114 binding_propensity=0.3453

KLGGALQAK
  HLA-A02:01   rank_el= 12.378  score_el=0.002508  non-binder
  HLA-A01:01   rank_el= 25.585  score_el=0.000580  non-binder
  HLA-B07:02   rank_el= 30.706  score_el=0.000188  non-binder
  summary: strong=0 weak=0 best_rank=12.378 binding_propensity=0.0119

AAGLQDCTM
  HLA-A02:01   rank_el= 36.583  score_el=0.000144  non-binder
  HLA-A01:01   rank_el= 18.989  score_el=0.000972  non-binder
  HLA-B07:02   rank_el= 10.511  score_el=0.002225 

In [6]:
# Human beta-2 microglobulin (mature chain, 99 residues) - the class I light chain itself.
B2M = (
    "IQRTPKIQVYSRHPAENGKSNFLNCYVSGFHPSDIEVDLLKNGERIEKVEHSDLSFSKDW"
    "SFYLLYYTEFTPTEKDEYACRVNHVTLSQPKIVKWDRDM"
)

# The protein tool does the windowing for you: every k-mer of every configured length is
# enumerated and scored. Here 9- and 10-mers against two alleles.
protein_inputs = NetMHCpanProteinInput(sequences=[B2M])
protein_config = NetMHCpanProteinConfig(
    alleles=["HLA-A02:01", "HLA-B07:02"],
    peptide_lengths=[9, 10],
)
protein_result = run_netmhcpan_protein(protein_inputs, protein_config)

scan = protein_result.results[0]
print(f"Sequence length: {scan.sequence_length} residues")
print(f"Windows scored:  {scan.num_windows}")
print(f"Binding records: {len(scan.bindings)}  (one per window per allele)")
print(
    f"Strong binders: {scan.strong_binder_count}, weak: {scan.weak_binder_count}, "
    f"best rank: {scan.best_rank:.3f}"
)
print(f"Strong binders per 100 residues per allele: {scan.strong_binders_per_100_residues:.2f}")

# The strongest predicted epitopes. `position` is 1-indexed against the supplied sequence.
print("\nTop 8 windows by EL %rank:")
for binding in sorted(scan.bindings, key=lambda b: b.rank_el)[:8]:
    print(
        f"  pos {binding.position:>3}  {binding.peptide:<11} {binding.allele:<12} "
        f"rank_el={binding.rank_el:>6.3f}  core={binding.core:<10} {binding.binder_class}"
    )

Running run_netmhcpan_protein [00:00]

Sequence length: 99 residues
Windows scored:  181
Binding records: 362  (one per window per allele)
Strong binders: 2, weak: 4, best rank: 0.218
Strong binders per 100 residues per allele: 1.01

Top 8 windows by EL %rank:
  pos  13  HPAENGKSNF  HLA-B07:02   rank_el= 0.218  core=HPAENGKSF  strong
  pos  31  HPSDIEVDL   HLA-B07:02   rank_el= 0.333  core=HPSDIEVDL  strong
  pos  31  HPSDIEVDLL  HLA-B07:02   rank_el= 0.991  core=HPSDIEVLL  weak
  pos  56  FSKDWSFYL   HLA-A02:01   rank_el= 1.188  core=FSKDWSFYL  weak
  pos  79  ACRVNHVTL   HLA-B07:02   rank_el= 1.285  core=ACRVNHVTL  weak
  pos  63  YLLYYTEFT   HLA-A02:01   rank_el= 1.344  core=YLLYYTEFT  weak
  pos  78  YACRVNHVTL  HLA-B07:02   rank_el= 2.144  core=YARVNHVTL  non-binder
  pos  30  FHPSDIEVDL  HLA-B07:02   rank_el= 2.473  core=FPSDIEVDL  non-binder


In [7]:
# Advanced usage: a wider allele panel, binding affinity alongside the default eluted-ligand
# output, and stricter binder thresholds.
#
#   include_binding_affinity=True passes netMHCpan's -BA flag, which *adds* score_ba, rank_ba
#   and affinity_nm to the EL columns rather than replacing them.
#   strong_rank / weak_rank only change how rank_el is classified; they do not change any score.
advanced_config = NetMHCpanPeptidesConfig(
    alleles=["HLA-A*02:01", "HLA-A01:01", "HLA-A03:01", "HLA-B07:02"],
    include_binding_affinity=True,
    strong_rank=0.2,
    weak_rank=1.0,
)
advanced_result = run_netmhcpan_peptides(inputs, advanced_config)

for prediction in advanced_result.predictions:
    print(f"\n{prediction.peptide}")
    for binding in prediction.bindings:
        print(
            f"  {binding.allele:<12} rank_el={binding.rank_el:>7.3f}  "
            f"rank_ba={binding.rank_ba:>7.3f}  affinity={binding.affinity_nm:>9.2f} nM  "
            f"{binding.binder_class}"
        )

# The BA fields are populated only when -BA was requested; in the default EL-only run above
# they are None. Same peptide, same allele, two runs:
default_binding = peptide_result.predictions[0].bindings[0]
ba_binding = advanced_result.predictions[0].bindings[0]
print(f"peptide={default_binding.peptide} allele={default_binding.allele}")
print(f"  EL-only run: rank_el={default_binding.rank_el}, rank_ba={default_binding.rank_ba}, "
      f"affinity_nm={default_binding.affinity_nm}")
print(f"  -BA run:     rank_el={ba_binding.rank_el}, rank_ba={ba_binding.rank_ba}, "
      f"affinity_nm={ba_binding.affinity_nm}")

# rank_el is identical across the two runs: -BA adds columns, it does not change the EL
# prediction. What did change is binder_class, because the thresholds were tightened.
print(f"\nrank_el unchanged: {default_binding.rank_el == ba_binding.rank_el}")
print(
    f"binder_class at strong_rank=0.5: {default_binding.binder_class}  ->  "
    f"at strong_rank=0.2: {ba_binding.binder_class}"
)

Running run_netmhcpan_peptides [00:00]


AAAWYLWEV
  HLA-A02:01   rank_el=  0.472  rank_ba=  0.095  affinity=     8.32 nM  weak
  HLA-A01:01   rank_el= 10.140  rank_ba=  3.647  affinity= 11442.33 nM  non-binder
  HLA-A03:01   rank_el= 23.207  rank_ba= 14.198  affinity= 18195.30 nM  non-binder
  HLA-B07:02   rank_el= 16.477  rank_ba= 12.238  affinity= 17781.15 nM  non-binder

GILGFVFTL
  HLA-A02:01   rank_el=  0.114  rank_ba=  0.077  affinity=     7.27 nM  strong
  HLA-A01:01   rank_el= 25.341  rank_ba= 39.180  affinity= 33480.61 nM  non-binder
  HLA-A03:01   rank_el=  7.633  rank_ba=  8.809  affinity= 12048.51 nM  non-binder
  HLA-B07:02   rank_el= 10.849  rank_ba= 23.261  affinity= 26148.13 nM  non-binder

KLGGALQAK
  HLA-A02:01   rank_el= 12.378  rank_ba= 24.743  affinity= 18955.51 nM  non-binder
  HLA-A01:01   rank_el= 25.585  rank_ba= 51.545  affinity= 36203.29 nM  non-binder
  HLA-A03:01   rank_el=  0.055  rank_ba=  0.213  affinity=    53.20 nM  strong
  HLA-B07:02   rank_el= 30.706  rank_ba= 36.565  affinity= 31922.65 

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# CSV gives a flat table, one row per peptide-allele (or window-allele) prediction - the form
# you want for downstream filtering and plotting.
advanced_result.export(name="netmhcpan_peptides", export_path=output_dir, file_format="csv")
protein_result.export(name="netmhcpan_protein_scan", export_path=output_dir, file_format="csv")

# JSON preserves the full nested structure: per-item summary metrics, every binding record,
# and the netMHCpan version that produced them.
advanced_result.export(name="netmhcpan_peptides", export_path=output_dir, file_format="json")

for path in sorted(output_dir.glob("netmhcpan_*")):
    print(f"{path.name}  ({path.stat().st_size} bytes)")

print("\nFirst rows of the peptide CSV:")
print("\n".join((output_dir / "netmhcpan_peptides.csv").read_text().splitlines()[:5]))

netmhcpan_peptides.csv  (1215 bytes)
netmhcpan_peptides.json  (7952 bytes)
netmhcpan_protein_scan.csv  (20116 bytes)

First rows of the peptide CSV:
peptide,allele,score_el,rank_el,binder_class,score_ba,rank_ba,affinity_nm
AAAWYLWEV,HLA-A02:01,0.440383,0.472,weak,0.804144,0.095,8.32
AAAWYLWEV,HLA-A01:01,0.002824,10.14,non-binder,0.136297,3.647,11442.33
AAAWYLWEV,HLA-A03:01,0.00027,23.207,non-binder,0.093427,14.198,18195.3
AAAWYLWEV,HLA-B07:02,0.000822,16.477,non-binder,0.095555,12.238,17781.15
